# New Zi init

*Version 2*

In [1]:
from zi import Zi

Good news: most of the "plumbing" is right (the `Complex` ABC registration, `__getitem__`-based unpacking, `__hash__`, `__complex__`). But there are several real bugs. Let me verify a few with actual execution rather than just eyeballing:That confirms several real bugs beyond the duplicate `__radd__`. Here's the priority list:

**1. `self._re` doesn't exist — it's `self._real`**

This breaks `__neg__`, `__pos__`, `conjugate`, and `__str__` (when `imag == 0`):

```python
def __neg__(self):
    return Zi(-self._real, -self.imag)   # was self._re

def __pos__(self):
    return Zi(self._real, self.imag)     # was self._re

def conjugate(self):
    return Zi(self._real, -self.imag)    # was self._re

def __str__(self):
    if self.imag == 0:
        return str(self._real)           # was self._re
    return str(complex(self.real, self.imag))
```

Since `Complex.__rsub__` (inherited from the ABC) is defined as `-self + other`, this bug silently breaks `2 - Zi(4,5)` too — that's the `rsub` failure above.

**2. `__rmul__`, `__truediv__`, `__rtruediv__`, `__pow__`, `__rpow__` are broken stubs** — same pattern as the `__radd__` issue. They all `return NotImplemented` unconditionally, and the division/pow ones are even missing the `other` parameter, so calling them directly would raise a `TypeError` about argument count. `2 * Zi(4,5)` fails for the same reason `2 + Zi(4,5)` did.

**3. `__eq__` raises instead of returning `NotImplemented`.** Python's data model expects rich comparisons to return `NotImplemented` for incomparable types, not raise:

```python
def __eq__(self, other):
    try:
        oth = Zi._ensure_zi(other)
    except TypeError:
        return NotImplemented
    return self.real == oth.real and self.imag == oth.imag
```

Without this, `z == "hello"` throws instead of just being `False`, and things like `z in some_list` or `assertEqual` against unrelated types will blow up.

**4. Hash/equality contract violation.** `_ensure_zi` lets `Zi(5, 0) == 5` be `True`, but `hash(Zi(5,0))` (`hash((5,0))`) ≠ `hash(5)`. Python requires equal objects to hash equal — violating this silently corrupts dict/set behavior. Either don't allow cross-type equality with plain numbers, or normalize the hash (e.g., hash as `complex(self)` when `imag == 0` matches a real).

Given you're fuzz-testing, I'd fix #1 and #3 immediately (they're outright crashes), then decide the intended semantics for #2 (should division return `Zi` rounded to nearest Gaussian integer, or should it produce a `Qi`?) before implementing. Want me to write out proper `__mul__`-consistent `__rmul__`, plus `__truediv__`/`__rtruediv__`/`__pow__`/`__rpow__` implementations?

```
def __rmul__(self, other):
    oth = Zi._ensure_zi(other)
    return oth * self          # multiplication is commutative

def __truediv__(self, other):
    oth = Zi._ensure_zi(other)
    n = oth.norm()
    if n == 0:
        raise ZeroDivisionError("division by zero Gaussian integer")
    num = self * oth.conjugate()          # (a+bi)(c-di) via existing __mul__
    return Zi(round(num.real / n), round(num.imag / n))

def __rtruediv__(self, other):
    oth = Zi._ensure_zi(other)
    return oth.__truediv__(self)

def __pow__(self, exponent):
    if not isinstance(exponent, int):
        return NotImplemented
    if exponent == 0:
        return Zi(1, 0)
    base, exp = (self, exponent) if exponent > 0 else (Zi(1, 0) / self, -exponent)
    result = Zi(1, 0)
    while exp > 0:
        if exp & 1:
            result = result * base
        base = base * base
        exp >>= 1
    return result

def __rpow__(self, base):
    if self.imag != 0:
        return NotImplemented          # exponent must be an ordinary integer
    return Zi._ensure_zi(base).__pow__(self.real)
```

Here's a consistent set, following the same pattern you used for `__add__`/`__radd__` (delegate to the primary op) and the standard Gaussian-integer division algorithm (divide, then round to nearest lattice point):

```python
def __rmul__(self, other):
    oth = Zi._ensure_zi(other)
    return oth * self          # multiplication is commutative

def __truediv__(self, other):
    oth = Zi._ensure_zi(other)
    n = oth.norm()
    if n == 0:
        raise ZeroDivisionError("division by zero Gaussian integer")
    num = self * oth.conjugate()          # (a+bi)(c-di) via existing __mul__
    return Zi(round(num.real / n), round(num.imag / n))

def __rtruediv__(self, other):
    oth = Zi._ensure_zi(other)
    return oth.__truediv__(self)

def __pow__(self, exponent):
    if not isinstance(exponent, int):
        return NotImplemented
    if exponent == 0:
        return Zi(1, 0)
    base, exp = (self, exponent) if exponent > 0 else (Zi(1, 0) / self, -exponent)
    result = Zi(1, 0)
    while exp > 0:
        if exp & 1:
            result = result * base
        base = base * base
        exp >>= 1
    return result

def __rpow__(self, base):
    if self.imag != 0:
        return NotImplemented          # exponent must be an ordinary integer
    return Zi._ensure_zi(base).__pow__(self.real)
```

Key design points, since these carry real math decisions rather than just plumbing:

- **`__truediv__` returns a `Zi`, not a `Qi` or `complex`.** Division isn't exact in `ℤ[i]`, so this rounds to the nearest Gaussian integer using the standard `a·b̄ / N(b)` trick, then `round()` on each coordinate (Python's `round()` is banker's-rounding, round-half-to-even). If you actually want exact quotients, that's what your `Qi` class is for — this operator is the "nearest lattice point" version, consistent with `modified_divmod`. Worth documenting clearly in the docstring so nobody's surprised that `Zi(1,0) / Zi(2,0)` gives `Zi(0,0)` or `Zi(1,0)` rather than a fraction.
- **`__pow__` supports negative integer exponents** by first inverting via `__truediv__` (so again, subject to rounding for non-unit bases) — e.g. `Zi(2,0) ** -1` won't be a true inverse since 2 isn't a unit in `ℤ[i]`. Only `±1, ±i` have exact inverses. Might be worth raising instead of silently rounding for negative exponents on non-units — your call depending on what the fuzz tests expect.
- **`__rpow__` only handles the case where the exponent (`self`) has `imag == 0`** — `Zi` exponents with nonzero imaginary part aren't well-defined in this ring, so it correctly bails to `NotImplemented` rather than guessing.

Let me verify these actually work end-to-end:All confirmed working end-to-end. A couple of things visible in that test output worth flagging:

- **`z ** -1 = 0`** — this isn't a bug, it's the expected consequence of rounding: `Zi(4,5)` isn't a unit in `ℤ[i]` (only `±1, ±i` have exact multiplicative inverses), so `1/z` rounds to the nearest lattice point, which happens to be `Zi(0,0)`. If your fuzz tests assert `z * (z**-1) == 1` for arbitrary `z`, that assertion will correctly fail for non-units — which might be exactly the kind of edge case you want the fuzz suite to catch, or might mean you want `__pow__` to raise on negative exponents unless `norm() == 1`. Worth deciding explicitly rather than letting it round silently.
- **`2 ** Zi(3, 0)` correctly falls back to `__rpow__`** and returns `8` — confirms the ABC dispatch chain (`int.__pow__` → `NotImplemented` → `Zi.__rpow__`) works the same way `__radd__` does.

One more thing since you're fuzz-testing heavily: your `__mul__`/`__truediv__` combo should satisfy `(a * b) / b == a` for any nonzero `b` where the division is exact (i.e., when `b | a` in `ℤ[i]`), and that's a good property to explicitly fuzz-test given the rounding behavior — it'll catch any sign or rounding-direction errors in the `a·b̄/N(b)` formula that a spot check like the one above might miss.

Glad it all checks out! Good luck with the rest of the `Gauss`/`Zi`/`Qi` work — feel free to come back when you're ready to fuzz-test `xgcd` or the `modified_divmod` rounding tie-points, since those tend to have trickier edge cases than the basic ring operations.

# Updates - Sat. July 11, 2026

I have added methods to the Zi class based on previous sessions and would like to have them verified and unittests written for them. The additional methods include: __isub__, __imul__, __ne__, __floordiv__, __rfloordiv__, __mod__, to_array, from_array, _is_rational_prime, is_gaussian_prime, modified_divmod, gcd, xgcd, random, eye, units, is_unit, and two. If modifications to the Zi class are recommended, please make the modifications and output an updated source file. Also, output a file of unittests that include all of the previous unittests along with new unittests for the additional methods. The source code and unittest code are attached below.
.....

## Answer

Let me look over each new method carefully before writing anything back.

**What I found:**

1. **`__floordiv__` has a precision bug.** It computes `complex(self) / complex(other)` using floats and then rounds. Your `__truediv__` instead does exact integer arithmetic (multiply by conjugate, divide by norm, then round) — which is precision-safe for arbitrarily large coefficients. Since `//` and `/` are meant to produce the *same* "nearest Gaussian integer" result in your design (there's no `Qi` here to hold the exact rational), having two different algorithms is a latent bug: for large coordinates, the float path in `__floordiv__` can round differently than the exact path in `__truediv__`, silently breaking `modified_divmod`, `gcd`, and `xgcd` (which rely on `//`). **Fix:** make `__floordiv__` just call `self.__truediv__(other)`.

2. **`is_gaussian_prime`/`is_real_prime` naming.** Your requested method list says `_is_rational_prime`, but the source has a public `is_real_prime`. I'll rename it to the private `_is_rational_prime` per your list (it's really an internal helper).

3. **`is_unit`** works but does a linear scan through a freshly-built 4-element list using `__eq__` each time. Since units of `Z[i]` are exactly the elements of norm 1, `self.norm() == 1` is equivalent and cheaper. I'll make that swap.

4. **`two()` returns `Zi(1, 1)`, not `Zi(2, 0)`.** That's `1+i`, the Gaussian prime lying above the ramified rational prime 2 (since `2 = -i(1+i)²`), not literally "the integer 2." Given your interest in ramification theory this looks intentional, but flagging it since the name is easy to misread — I'm leaving it as-is. Let me know if you actually wanted `Zi(2, 0)`.

5. Everything else (`__isub__`, `__imul__`, `__ne__`, `__mod__`, `to_array`/`from_array`, `is_gaussian_prime`, `modified_divmod`, `gcd`, `xgcd`, `random`, `eye`, `units`) checked out mathematically — `is_gaussian_prime`'s `p ≡ 3 (mod 4)` branch correctly excludes the ramified prime 2 and split primes `≡ 1 mod 4`, and `xgcd`'s Bézout recurrence is standard and correct given a fixed `modified_divmod`.

Let me write the updated source and full test suite.Now let's actually run the tests to make sure everything checks out.All 123 tests pass. Also verified it demonstrated the actual bug I found: I proved `__floordiv__`'s old float-based path could disagree with `__truediv__` on large coefficients — `test_division_precise_for_large_coefficients` / `test_floordiv_precise_for_large_coefficients` exercise exactly that (using coefficients around `10^8`–`10^9`, where floating-point complex division starts losing bits your exact conjugate-based method doesn't).**Summary of changes to `zi.py`:**

1. **`__floordiv__` fixed** — now delegates to `__truediv__` instead of doing float-based `complex(self) / complex(other)`. This closes a real bug: for large coefficients the float path could round to a different Gaussian integer than the exact conjugate/norm path, which would have silently corrupted `modified_divmod`, `gcd`, and `xgcd`.
2. **`is_real_prime` → `_is_rational_prime`**, matching your requested method name and making clear it's an internal helper, not a statement about Gaussian primality.
3. **`is_unit`** now checks `self.norm() == 1` instead of scanning `Zi.units()` — same result, no temp list, no `__eq__` calls.
4. **`two()`** left as `Zi(1, 1)` (the prime above the ramified rational prime 2) — flagged above, not changed, since it looked deliberate given your ramification-theory work.

**Test file** (123 tests, all passing) adds coverage for every method on your list — `__isub__`/`__imul__` (including immutability checks), `__ne__` (including `NotImplemented` propagation), `__floordiv__`/`__rfloordiv__`/`__mod__` (including a `floordiv == truediv` invariant check and large-coefficient precision), `to_array`/`from_array`, `_is_rational_prime`, `is_gaussian_prime` (covering the ramified/split/inert cases explicitly — 2 not prime, 5 not prime but 2±i prime, 3 and 7 prime), `modified_divmod`/`gcd`/`xgcd` (with Bézout identity and remainder-norm-decrease checks), and `random`/`eye`/`units`/`is_unit`/`two`, plus fuzz extensions of the algebraic-law tests to the new operators.